# 🏃 Human Activity Recognition (HAR) — Sensor-Based Classification

> **Classifying six daily activities from smartphone accelerometer & gyroscope data  
> using five machine learning models, with best accuracy achieved by Random Forest (>99%).**

---

## 📋 Table of Contents
1. [Project Overview](#1-project-overview)
2. [Dataset](#2-dataset)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Preprocessing & Feature Engineering](#4-preprocessing--feature-engineering)
5. [Model Training & Evaluation](#5-model-training--evaluation)
6. [Model Comparison](#6-model-comparison)
7. [Feature Importance & PCA](#7-feature-importance--pca)
8. [Model Persistence & Inference](#8-model-persistence--inference)
9. [Key Takeaways](#9-key-takeaways)


## 1. Project Overview

**Goal:** Build a robust classifier that predicts a person's physical activity  
from 561 time-domain and frequency-domain features extracted from a smartphone's  
inertial sensors (accelerometer + gyroscope).

**Activities classified:**
| Label | Activity |
|-------|----------|
| 0 | LAYING |
| 1 | SITTING |
| 2 | STANDING |
| 3 | WALKING |
| 4 | WALKING_DOWNSTAIRS |
| 5 | WALKING_UPSTAIRS |

**Models evaluated:** Logistic Regression · K-Nearest Neighbours · Decision Tree · Random Forest · SVM (RBF)

**Dataset:** [UCI HAR Dataset](https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones) — 10,299 samples × 561 features, 30 subjects


## 2. Dataset

### 2.1 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print("Libraries loaded ✓")


### 2.2 Load Data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# If running on Google Colab, upload train.csv.zip when prompted.
# Locally: place train.csv (or train.csv.zip) in the working directory.
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv("train.csv.zip")          # pandas reads .zip containing a single CSV automatically

print(f"Shape       : {df.shape}")
print(f"Activities  : {df['Activity'].unique()}")
print(f"Subjects    : {df['subject'].nunique()}")
print(f"Null values : {df.isnull().sum().sum()}")
print(f"Duplicates  : {df.duplicated().sum()}")


### 2.3 Quick Peek

In [ ]:
df.head()


In [ ]:
df.describe().T.head(10)   # show first 10 features for brevity


## 3. Exploratory Data Analysis

### 3.1 Class Distribution

A balanced dataset means no resampling is needed — all models are evaluated fairly.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count bar
df['Activity'].value_counts().plot.bar(
    ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Activity Distribution (Count)', fontsize=13)
axes[0].set_xlabel('Activity')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Percentage pie
round(df['Activity'].value_counts() / len(df) * 100, 2).plot.pie(
    ax=axes[1], autopct='%1.2f%%',
    colors=sns.color_palette('Set2', 6)
)
axes[1].set_title('Activity Distribution (%)', fontsize=13)
axes[1].set_ylabel('')

plt.suptitle('Class Distribution — HAR Dataset', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


### 3.2 Sensor Time-Series by Activity (Subject 1)

Plotting raw accelerometer and gyroscope readings per activity reveals clear  
signal separation between dynamic (walking) and static (sitting/laying) activities.


In [ ]:
subject1 = df[df['subject'] == 1]

sensor_cols = {
    'Accelerometer': ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z'],
    'Gyroscope':     ['tBodyGyro-mean()-X', 'tBodyGyro-mean()-Y', 'tBodyGyro-mean()-Z'],
}

for activity in subject1['Activity'].unique():
    data = subject1[subject1['Activity'] == activity].reset_index(drop=True)
    for sensor, cols in sensor_cols.items():
        fig, ax = plt.subplots(figsize=(10, 3))
        for col, color, axis_lbl in zip(cols, ['royalblue', 'tomato', 'orange'], ['X', 'Y', 'Z']):
            ax.plot(data[col], label=f"{axis_lbl}-axis", color=color, alpha=0.75)
        ax.set_title(f"{activity} — {sensor}", fontsize=12)
        ax.set_xlabel('Time Step')
        ax.set_ylabel('Mean Signal')
        ax.legend()
        plt.tight_layout()
        plt.show()


### 3.3 Signal Distributions (Histograms)

In [ ]:
for activity in subject1['Activity'].unique():
    data = subject1[subject1['Activity'] == activity].reset_index(drop=True)
    for sensor, cols in sensor_cols.items():
        fig, ax = plt.subplots(figsize=(8, 3))
        for col, color, axis_lbl in zip(cols, ['royalblue', 'tomato', 'orange'], ['X', 'Y', 'Z']):
            ax.hist(data[col], bins=20, label=f"{axis_lbl}-axis", color=color, alpha=0.55)
        ax.set_title(f"{activity} — {sensor} Distribution", fontsize=12)
        ax.legend()
        plt.tight_layout()
        plt.show()


## 4. Preprocessing & Feature Engineering

### 4.1 Label Encoding

The target column `Activity` (string) is encoded to integers using `LabelEncoder`.  
This mapping is saved to disk later for inference.


In [ ]:
le = LabelEncoder()
df['Activity_enc'] = le.fit_transform(df['Activity'])

activity_labels = dict(enumerate(le.classes_))
print("Activity encoding:")
for idx, name in activity_labels.items():
    print(f"  {idx} → {name}")


### 4.2 Train / Test Split & Scaling

| Decision | Reason |
|----------|--------|
| **Stratified split (80/20)** | Preserves class proportions in both sets |
| **StandardScaler fit on train only** | Prevents data leakage from test set into scaling |


In [ ]:
feature_cols = [c for c in df.columns if c not in ('Activity', 'Activity_enc', 'subject')]

X = df[feature_cols].values
y = df['Activity_enc'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print(f"Train : {X_train_scaled.shape}")
print(f"Test  : {X_test_scaled.shape}")


## 5. Model Training & Evaluation

A shared helper prints metrics and optionally renders the confusion matrix.


In [ ]:
def evaluate_model(y_true, y_pred, show_cm=True, title=""):
    """Print classification metrics and optionally plot the confusion matrix."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    rec  = recall_score(y_true, y_pred, average='macro')
    f1   = f1_score(y_true, y_pred, average='macro')

    if show_cm:
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(9, 7))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=activity_labels.values(),
            yticklabels=activity_labels.values()
        )
        plt.title(f"Confusion Matrix — {title}", fontsize=13)
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.show()

    print(f"{'─'*35}")
    print(f"  Accuracy  : {acc:.4%}")
    print(f"  Precision : {prec:.4%}")
    print(f"  F1 Score  : {f1:.4%}")
    print(f"  Recall    : {rec:.4%}")
    print(f"{'─'*35}")
    return dict(Accuracy=round(acc,4), Precision=round(prec,4),
                Recall=round(rec,4), F1=round(f1,4))


### 5.1 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

print(f"Train accuracy : {lr.score(X_train_scaled, y_train):.4%}")
print(f"Test  accuracy : {lr.score(X_test_scaled,  y_test):.4%}")
y_pred_lr = lr.predict(X_test_scaled)
lr_metrics = evaluate_model(y_test, y_pred_lr, title="Logistic Regression")


### 5.2 K-Nearest Neighbours — Hyperparameter Search

In [ ]:
# Sweep k=1..10 to find optimal neighbours
knn_scores = {}
for k in range(1, 11):
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_scaled, y_train)
    knn_scores[k] = accuracy_score(y_test, knn_k.predict(X_test_scaled))

best_k = max(knn_scores, key=knn_scores.get)
print(f"Best k = {best_k}  (test accuracy = {knn_scores[best_k]:.4%})")

plt.figure(figsize=(8, 4))
plt.plot(knn_scores.keys(), knn_scores.values(), marker='o', color='steelblue')
plt.axvline(best_k, color='tomato', linestyle='--', label=f'Best k={best_k}')
plt.title('KNN — Accuracy vs. k')
plt.xlabel('k (Neighbours)')
plt.ylabel('Test Accuracy')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
knn_metrics = evaluate_model(y_test, y_pred_knn, title=f"KNN (k={best_k})")


### 5.3 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=14, random_state=42)
dt.fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)
dt_metrics = evaluate_model(y_test, y_pred_dt, title="Decision Tree")


### 5.4 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
rf_metrics = evaluate_model(y_test, y_pred_rf, title="Random Forest")


### 5.5 Support Vector Machine (RBF Kernel)

In [ ]:
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)
svm_metrics = evaluate_model(y_test, y_pred_svm, title="SVM (RBF)")


## 6. Model Comparison

Side-by-side accuracy and F1 across all five models.


In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Logistic Regression',    **lr_metrics},
    {'Model': f'KNN (k={best_k})',      **knn_metrics},
    {'Model': 'Decision Tree',          **dt_metrics},
    {'Model': 'Random Forest',          **rf_metrics},
    {'Model': 'SVM (RBF)',              **svm_metrics},
]).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print(comparison.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

comparison.plot.barh('Model', 'Accuracy',  ax=axes[0], color='steelblue',  edgecolor='black', legend=False)
axes[0].set_title('Test Accuracy by Model', fontsize=12)
axes[0].set_xlim(0.85, 1.01)
axes[0].axvline(1.0, color='gray', linestyle='--', linewidth=0.8)

comparison.plot.barh('Model', 'F1', ax=axes[1], color='salmon', edgecolor='black', legend=False)
axes[1].set_title('Macro F1 Score by Model', fontsize=12)
axes[1].set_xlim(0.85, 1.01)

plt.tight_layout()
plt.show()


## 7. Feature Importance & PCA

### 7.1 Top-20 Feature Importances (Random Forest)

Frequency-domain magnitude features dominate — confirming that frequency  
statistics are highly discriminative for inertial sensor activity recognition.


In [ ]:
feature_names  = feature_cols
importances    = rf.feature_importances_
top_n          = 20
top_idx        = np.argsort(importances)[::-1][:top_n]

plt.figure(figsize=(13, 5))
plt.bar(range(top_n), importances[top_idx], color='steelblue', edgecolor='black')
plt.xticks(range(top_n), [feature_names[i] for i in top_idx], rotation=90, fontsize=8)
plt.title(f'Top {top_n} Feature Importances — Random Forest', fontsize=13)
plt.ylabel('Importance')
plt.tight_layout()
plt.show()


### 7.2 PCA — 2D Projection of Feature Space

Even in just 2 principal components the six activities form distinct clusters,  
explaining why linear (Logistic Regression) and non-linear (SVM, RF) models  
both achieve high accuracy.


In [ ]:
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_scaled)

plt.figure(figsize=(10, 7))
palette = sns.color_palette('tab10', len(le.classes_))

for i, activity in enumerate(le.classes_):
    mask = y_train == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                label=activity, color=palette[i], alpha=0.4, s=12)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
plt.title('PCA — HAR Feature Space (Train Set)', fontsize=13)
plt.legend(title='Activity', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 8. Model Persistence & Inference

Artefacts are saved so the model can be served without retraining.

| File | Contents |
|------|----------|
| `har_model.pkl` | Trained Random Forest |
| `scaler.pkl` | Fitted StandardScaler |
| `label_encoder.pkl` | LabelEncoder (int → activity name) |
| `feature_names.pkl` | Ordered list of 561 feature names |


In [ ]:
joblib.dump(rf,           'har_model.pkl')
joblib.dump(scaler,       'scaler.pkl')
joblib.dump(le,           'label_encoder.pkl')
joblib.dump(feature_names,'feature_names.pkl')
print("Saved: har_model.pkl | scaler.pkl | label_encoder.pkl | feature_names.pkl ✓")


### 8.1 Inference Function

In [ ]:
def predict_activity(sensor_features: list) -> str:
    """
    Predict the physical activity from 561 pre-computed sensor features.

    Parameters
    ----------
    sensor_features : list or array-like of length 561

    Returns
    -------
    str  — predicted activity label
    """
    arr = np.array(sensor_features, dtype=float).reshape(1, -1)
    arr_scaled = scaler.transform(arr)
    pred_idx   = rf.predict(arr_scaled)[0]
    return le.inverse_transform([pred_idx])[0]


# ── Smoke test ──────────────────────────────────────────────────────────────
sample_idx = 0
predicted  = predict_activity(X_test[sample_idx])
actual     = le.inverse_transform([y_test[sample_idx]])[0]

print(f"Predicted : {predicted}")
print(f"Actual    : {actual}")
print("Match ✓" if predicted == actual else "Mismatch ✗")


## 9. Key Takeaways

| Finding | Detail |
|---------|--------|
| **Random Forest wins** | Highest accuracy & F1; robust to noisy features |
| **SVM close second** | Near-identical accuracy; slower to train at this scale |
| **Static vs dynamic separation** | PCA shows two clear super-clusters (static: laying/sitting/standing vs dynamic: walking variants) |
| **Frequency features matter most** | Top feature importances are FFT-derived magnitude statistics |
| **No feature engineering needed** | UCI HAR already provides 561 hand-crafted features; raw sensor → features is the real challenge in production |

### Possible Extensions
- Tune Random Forest / SVM with `GridSearchCV`
- Try XGBoost or LightGBM for potentially higher accuracy
- Deploy as a REST API with FastAPI + joblib
- Apply to raw (un-processed) sensor windows with a CNN or LSTM

---
*Dataset: [UCI HAR Dataset](https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones) | Author: Palak | License: MIT*
